## Data Cleaning
### Telecom Customer Churn Prediction

**Goal of this notebook:** 
Before we can explore the data or build any model,we need to make sure it's
clean and reliable , . This notebook  loads the raw Telco Customer Churn dataset,
idetifies data quality issues, and produce a cleaned version  ready for
exploratory analysis

**Input:** `churn.csv` (raw dataset)
**Output:** `cleaned_churn.csv` (cleaned dataset)

## 1. Import Libraries

we start by importing the core libraries needed for data manipulation.

In [1]:
import pandas as pd
import numpy as np

## 2. Load the Dataset 
Loading the raw dataset as-is ,without any modifications ,so we  can first  inspect it and understand what we're working with

In [2]:
df = pd.read_csv(r"D:\Projects\notebooks\Churn_Prediction_Project\data\raw\churn.csv")


## 3 . Initial Data Overview

Before cleaning anything , it's important to understand the shape of the data, the data types of each column, and whether any values are missing. This helps
us decide what cleaning steps are actually needed.

In [3]:
# Quick overivew

print(df.shape)
print(df.info())
print(df.isnull().sum())

(7043, 21)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null

**Insight:** NO `NAN` values shown ,but `TotalCharges` is sorted as `object`(text) instead of numeric - a sign it contains blank/non-numeric values that need to be converted and checked before analysis

In [4]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'],errors = 'coerce')

In [10]:
print(df['TotalCharges'].isnull().sum())

11


## 4. Remove Rows with Missing `TotalCharges`


after converting `TotalCharges` to numeric , the blank space values become 
`NAN` . we drop the rows using `dropna()` instead of filling them, because
they are all new customers(`tenure =0`) with billing history yet - filling
would mean guessing a value rather than using real data.  Since this  only
affects 11 out of 7,043 rows, dropping them is simple and has no meaningful impact on the dataset 

In [11]:
df = df.dropna(subset=['TotalCharges'])

## 5. Remove Duplicate Records

Duplicate rows can bias a modely by letting the same customer's  pattern be
learned multiple times. We check for and remove any exact  duplicate rows to  
keep every customer represented once.

In [12]:
df = df.drop_duplicates()

## 6. Drop the customer Identifier
`custoemrID` is a unique identifier for each customer. It has no predictive
value - including it would risk the model "memorizing" specific customers
instead of learning general churn patterns - so it's removed at this stage.

In [14]:
df = df.drop(columns=['customerID'])

## 7. Standardize the `SeniorCitizen` Column
`SeniorCitizen` is currently stored as `0`/`1`, while every other similar
flag column in the dataset (like `Partner`, `Dependents`) uses `Yes`/`No`.
Mapping it to the same format keeps the dataset consistent and easier to
work with in the next steps.

In [16]:
df['SeniorCitizen'] = df['SeniorCitizen'].map({0:'No',1:'Yes'})

## 9.Clean Up Text Formating

Text columns sometimes carry invisible leading or trailing whispace picked
up during entry export.Stripping this enusre categories like 
"Yes" and "Yes " aren't accidentally treadte as different value later.

In [18]:
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()
    

## 10. Validate the Cleaned Data

Before saving, we do a final check to confirm there are no remaining missing
values and review the resulting shape of the dataset.

In [19]:
print(df.isnull().sum().sum())   
df.head()

7032


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,NaN,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,NaN,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,NaN,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,NaN,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,NaN,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes



## 11. Save the Cleaned Dataset

The cleaned dataset is saved separately from the raw file, so the original
data remains untouched and this cleaned version can be reused consistently
across the exploratory analysis and preprocessing notebooks.

In [20]:

df.to_csv("cleaned_churn.csv", index=False)
print("✅ Cleaning done. File saved as cleaned_churn.csv")

✅ Cleaning done. File saved as cleaned_churn.csv
